In [44]:
from google.colab import files
import nbformat
from io import BytesIO

# Step 1: Upload your notebook
uploaded = files.upload()

for filename, filedata in uploaded.items():
    print(f"Uploaded notebook: {filename}")

    # Step 2: Read notebook as version 4 (Colab notebooks are v4)
    nb = nbformat.reads(filedata.decode("utf-8"), as_version=4)

    # Step 3: Remove widgets metadata if it exists
    if "widgets" in nb.metadata:
        del nb.metadata["widgets"]
        print("Removed widgets metadata!")

    # Step 4: Save fixed notebook
    fixed_filename = "fixed_" + filename
    with open(fixed_filename, "w", encoding="utf-8") as f:
        nbformat.write(nb, f)

    # Step 5: Download fixed notebook automatically
    files.download(fixed_filename)
    print(f"Fixed notebook downloaded as: {fixed_filename}")

KeyboardInterrupt: 

In [2]:
!pip install transformers datasets seqeval torch

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 1.7 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for seqeval: filename=seqeval-1.2.2-py3-none-any.whl size=16162 sha256=0c5b8ee093f28c6e3b58c1a66057139ca0e26d6ff923169e6a2263ee4d58cdd0
  Stored in directory: /root/.cache/pip/wheels/5f/b8/73/0b2c1a76b701a677653dd79ece07cfabd7457989dbfbdcd8d7
Successfully built seqeval


In [3]:
import numpy as np
import torch
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForTokenClassification
from transformers import TrainingArguments, Trainer
from seqeval.metrics import precision_score, recall_score, f1_score, classification_report

In [6]:
import zipfile

zip_path = "/content/CoNLL2003.zip"   # change filename if needed
extract_path = "/content/CoNLL2003"

with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(extract_path)

print("Unzipped successfully")

Unzipped successfully


In [7]:
def read_conll(file_path):

    sentences = []
    pos_tags = []
    chunk_tags = []

    words = []
    pos = []
    chunk = []

    with open(file_path, "r", encoding="utf-8") as f:

        for line in f:

            if line.strip() == "":

                if words:
                    sentences.append(words)
                    pos_tags.append(pos)
                    chunk_tags.append(chunk)

                    words = []
                    pos = []
                    chunk = []

            else:

                parts = line.split()

                words.append(parts[0])

                pos.append(parts[1])

                chunk.append(parts[2])

    return sentences, pos_tags, chunk_tags

Load train, validation, test

In [8]:
train_sentences, train_pos, train_chunk = read_conll(
    "CoNLL2003/conll2003/eng.train"
)

val_sentences, val_pos, val_chunk = read_conll(
    "CoNLL2003/conll2003/eng.testa"
)

test_sentences, test_pos, test_chunk = read_conll(
    "CoNLL2003/conll2003/eng.testb"
)

Task 1 Deliverables
Label categories

In [9]:
unique_pos = sorted(list(set(tag for sent in train_pos for tag in sent)))

unique_chunk = sorted(list(set(tag for sent in train_chunk for tag in sent)))

print("POS labels:", unique_pos)

print("Chunk labels:", unique_chunk)

POS labels: ['"', '$', "''", '(', ')', ',', '-X-', '.', ':', 'CC', 'CD', 'DT', 'EX', 'FW', 'IN', 'JJ', 'JJR', 'JJS', 'LS', 'MD', 'NN', 'NNP', 'NNPS', 'NNS', 'NN|SYM', 'PDT', 'POS', 'PRP', 'PRP$', 'RB', 'RBR', 'RBS', 'RP', 'SYM', 'TO', 'UH', 'VB', 'VBD', 'VBG', 'VBN', 'VBP', 'VBZ', 'WDT', 'WP', 'WP$', 'WRB']
Chunk labels: ['-X-', 'B-ADJP', 'B-ADVP', 'B-CONJP', 'B-INTJ', 'B-LST', 'B-NP', 'B-PP', 'B-PRT', 'B-SBAR', 'B-VP', 'I-ADJP', 'I-ADVP', 'I-CONJP', 'I-INTJ', 'I-LST', 'I-NP', 'I-PP', 'I-SBAR', 'I-VP', 'O']


Task 2: Convert labels → IDs

In [10]:
pos2id = {tag:i for i,tag in enumerate(unique_pos)}

chunk2id = {tag:i for i,tag in enumerate(unique_chunk)}

id2pos = {i:tag for tag,i in pos2id.items()}

id2chunk = {i:tag for tag,i in chunk2id.items()}

Encode labels

In [11]:
# combine all labels first

all_pos = train_pos + val_pos + test_pos

all_chunk = train_chunk + val_chunk + test_chunk


unique_pos = sorted(
    list(set(tag for sent in all_pos for tag in sent))
)


unique_chunk = sorted(
    list(set(tag for sent in all_chunk for tag in sent))
)


print("POS labels:", unique_pos)

print("Chunk labels:", unique_chunk)

POS labels: ['"', '$', "''", '(', ')', ',', '-X-', '.', ':', 'CC', 'CD', 'DT', 'EX', 'FW', 'IN', 'JJ', 'JJR', 'JJS', 'LS', 'MD', 'NN', 'NNP', 'NNPS', 'NNS', 'NN|SYM', 'PDT', 'POS', 'PRP', 'PRP$', 'RB', 'RBR', 'RBS', 'RP', 'SYM', 'TO', 'UH', 'VB', 'VBD', 'VBG', 'VBN', 'VBP', 'VBZ', 'WDT', 'WP', 'WP$', 'WRB']
Chunk labels: ['-X-', 'B-ADJP', 'B-ADVP', 'B-CONJP', 'B-INTJ', 'B-LST', 'B-NP', 'B-PP', 'B-PRT', 'B-SBAR', 'B-VP', 'I-ADJP', 'I-ADVP', 'I-CONJP', 'I-INTJ', 'I-LST', 'I-NP', 'I-PP', 'I-PRT', 'I-SBAR', 'I-VP', 'O']


In [12]:
pos2id = {tag:i for i,tag in enumerate(unique_pos)}

chunk2id = {tag:i for i,tag in enumerate(unique_chunk)}

id2pos = {i:tag for tag,i in pos2id.items()}

id2chunk = {i:tag for tag,i in chunk2id.items()}

In [13]:
def encode_labels(sent_labels, label2id):
    encoded = []
    for sentence in sent_labels:
        encoded.append([label2id[tag] for tag in sentence])
    return encoded

train_pos_ids = encode_labels(train_pos, pos2id)
val_pos_ids = encode_labels(val_pos, pos2id)
test_pos_ids = encode_labels(test_pos, pos2id)

train_chunk_ids = encode_labels(train_chunk, chunk2id)
val_chunk_ids = encode_labels(val_chunk, chunk2id)
test_chunk_ids = encode_labels(test_chunk, chunk2id)

Tokenization (BERT)

In [14]:
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Label Alignment (handle subwords)

In [15]:
def tokenize_and_align(sentences, labels):

    encodings = tokenizer(

        sentences,

        is_split_into_words=True,

        padding=True,

        truncation=True,

        max_length=128

    )

    aligned_labels = []

    for i, label in enumerate(labels):

        word_ids = encodings.word_ids(batch_index=i)

        label_ids = []

        previous_word_idx = None

        for word_idx in word_ids:

            if word_idx is None:

                label_ids.append(-100)

            elif word_idx != previous_word_idx:

                label_ids.append(label[word_idx])

            else:

                label_ids.append(-100)

            previous_word_idx = word_idx

        aligned_labels.append(label_ids)

    encodings["labels"] = aligned_labels

    return encodings

Apply preprocessing
POS

In [16]:
train_pos_encodings = tokenize_and_align(
    train_sentences,
    train_pos_ids
)

val_pos_encodings = tokenize_and_align(
    val_sentences,
    val_pos_ids
)

test_pos_encodings = tokenize_and_align(
    test_sentences,
    test_pos_ids
)

Chunking

In [17]:
train_chunk_encodings = tokenize_and_align(
    train_sentences,
    train_chunk_ids
)

val_chunk_encodings = tokenize_and_align(
    val_sentences,
    val_chunk_ids
)

test_chunk_encodings = tokenize_and_align(
    test_sentences,
    test_chunk_ids
)

Convert to PyTorch Dataset

In [18]:
class TokenDataset(torch.utils.data.Dataset):

    def __init__(self, encodings):

        self.encodings = encodings

    def __getitem__(self, idx):

        item = {
            key: torch.tensor(val[idx])
            for key, val in self.encodings.items()
        }

        return item

    def __len__(self):

        return len(self.encodings["input_ids"])

In [38]:
train_pos_dataset = TokenDataset(train_pos_encodings)
val_pos_dataset = TokenDataset(val_pos_encodings)

train_chunk_dataset = TokenDataset(train_chunk_encodings)
val_chunk_dataset = TokenDataset(val_chunk_encodings)


small_train_pos_dataset = torch.utils.data.Subset(train_pos_dataset, range(2000))
small_val_pos_dataset = torch.utils.data.Subset(val_pos_dataset, range(400))

small_train_chunk_dataset = torch.utils.data.Subset(train_chunk_dataset, range(2000))
small_val_chunk_dataset = torch.utils.data.Subset(val_chunk_dataset, range(400))

In [ ]:
# train_pos_dataset = TokenDataset(train_pos_encodings)

# val_pos_dataset = TokenDataset(val_pos_encodings)

# test_pos_dataset = TokenDataset(test_pos_encodings)


# train_chunk_dataset = TokenDataset(train_chunk_encodings)

# val_chunk_dataset = TokenDataset(val_chunk_encodings)

# test_chunk_dataset = TokenDataset(test_chunk_encodings)

Task 3: Model Setup
POS Model

In [20]:
pos_model = AutoModelForTokenClassification.from_pretrained(

    "bert-base-uncased",

    num_labels=len(unique_pos),

    id2label=id2pos,

    label2id=pos2id
)

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

BertForTokenClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
bert.pooler.dense.weight                   | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
bert.pooler.dense.bias                     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized be

Chunk Model

In [39]:
chunk_model = AutoModelForTokenClassification.from_pretrained(

    "bert-base-uncased",

    num_labels=len(unique_chunk),

    id2label=id2chunk,

    label2id=chunk2id
)

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

BertForTokenClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
bert.pooler.dense.weight                   | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
bert.pooler.dense.bias                     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized be

Task 4: Training Settings

In [22]:
import transformers
print(transformers.__version__)

5.0.0


In [ ]:
!pip install --upgrade transformers

In [23]:
from transformers import TrainingArguments

In [40]:
training_args = TrainingArguments(

    output_dir="results",

    learning_rate=2e-5,

    per_device_train_batch_size=8,

    per_device_eval_batch_size=8,

    num_train_epochs=1,

    eval_strategy="epoch"
)

Task 5: Evaluation Metrics (seqeval)

In [25]:
from sklearn.metrics import precision_recall_fscore_support

def compute_pos_metrics(pred):

    predictions = np.argmax(pred.predictions, axis=2)

    labels = pred.label_ids

    true_preds = []
    true_labels = []

    for pred_seq, label_seq in zip(predictions, labels):

        for p, l in zip(pred_seq, label_seq):

            if l != -100:

                true_preds.append(unique_pos[p])

                true_labels.append(unique_pos[l])

    precision, recall, f1, _ = precision_recall_fscore_support(
        true_labels,
        true_preds,
        average="weighted"
    )

    return {

        "precision": precision,

        "recall": recall,

        "f1": f1
    }

In [ ]:
# def compute_metrics(pred, label_list):

#     predictions = np.argmax(pred.predictions, axis=2)

#     labels = pred.label_ids

#     true_predictions = [

#         [label_list[p] for (p,l) in zip(pred, lab) if l != -100]

#         for pred, lab in zip(predictions, labels)
#     ]

#     true_labels = [

#         [label_list[l] for (p,l) in zip(pred, lab) if l != -100]

#         for pred, lab in zip(predictions, labels)
#     ]

#     return {

#         "precision": precision_score(true_labels, true_predictions),

#         "recall": recall_score(true_labels, true_predictions),

#         "f1": f1_score(true_labels, true_predictions)
#     }

Train POS Model

In [26]:
from transformers import AutoModelForTokenClassification

pos_model = AutoModelForTokenClassification.from_pretrained(

    "bert-base-uncased",

    num_labels=len(unique_pos),

    id2label=id2pos,

    label2id=pos2id
)

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

BertForTokenClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
bert.pooler.dense.weight                   | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
bert.pooler.dense.bias                     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized be

In [41]:
def compute_chunk_metrics(pred):

    predictions = np.argmax(pred.predictions, axis=2)

    labels = pred.label_ids

    true_preds = [
        [unique_chunk[p] for p,l in zip(pred_seq, label_seq) if l != -100]
        for pred_seq, label_seq in zip(predictions, labels)
    ]

    true_labels = [
        [unique_chunk[l] for p,l in zip(pred_seq, label_seq) if l != -100]
        for pred_seq, label_seq in zip(predictions, labels)
    ]

    return {

        "precision": precision_score(true_labels, true_preds),

        "recall": recall_score(true_labels, true_preds),

        "f1": f1_score(true_labels, true_preds)
    }

Evaluate POS Model

In [28]:
pos_trainer = Trainer(

    model=pos_model,

    args=training_args,

    train_dataset=small_train_pos_dataset,

    eval_dataset=small_val_pos_dataset,

    compute_metrics=compute_pos_metrics
)

pos_trainer.train()

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Epoch,Training Loss,Validation Loss,Precision,Recall,F1
1,No log,0.672907,0.837546,0.863549,0.847280


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


TrainOutput(global_step=250, training_loss=1.4969622802734375, metrics={'train_runtime': 2909.6803, 'train_samples_per_second': 0.687, 'train_steps_per_second': 0.086, 'total_flos': 130700350464000.0, 'train_loss': 1.4969622802734375, 'epoch': 1.0})

Train Chunk Model

Evaluate Chunk Model

In [42]:
chunk_trainer = Trainer(

    model=chunk_model,

    args=training_args,

    train_dataset=small_train_chunk_dataset,

    eval_dataset=small_val_chunk_dataset,

    compute_metrics=compute_chunk_metrics
)

chunk_trainer.train()

Epoch,Training Loss,Validation Loss


Epoch,Training Loss,Validation Loss,Precision,Recall,F1
1,No log,0.473975,0.810407,0.798484,0.804402


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
/usr/local/lib/python3.12/dist-packages/seqeval/metrics/sequence_labeling.py:171: UserWarning: -X- seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))


TrainOutput(global_step=250, training_loss=0.8036581420898438, metrics={'train_runtime': 2985.1894, 'train_samples_per_second': 0.67, 'train_steps_per_second': 0.084, 'total_flos': 130672002048000.0, 'train_loss': 0.8036581420898438, 'epoch': 1.0})

Task 6: Inference on Custom Sentence

POS Prediction

In [35]:
sentence = "John works at Google in California"

tokens = tokenizer(
    sentence.split(),
    is_split_into_words=True,
    return_tensors="pt",
    padding=True,
    truncation=True
)

pos_outputs = pos_model(**tokens).logits

pos_preds = torch.argmax(pos_outputs, dim=2)

word_ids = tokens.word_ids()

predicted_tags = []

previous_word_idx = None

for pred, word_idx in zip(pos_preds[0], word_ids):

    if word_idx is None:
        continue

    if word_idx != previous_word_idx:

        predicted_tags.append(id2pos[pred.item()])

    previous_word_idx = word_idx


for word, tag in zip(sentence.split(), predicted_tags):

    print(word, "→", tag)

John → NNP
works → VBZ
at → IN
Google → NNP
in → IN
California → NNP


Chunk Prediction

In [43]:
chunk_outputs = chunk_model(**tokens).logits

chunk_preds = torch.argmax(chunk_outputs, dim=2)

word_ids = tokens.word_ids()

predicted_chunks = []

previous_word_idx = None

for pred, word_idx in zip(chunk_preds[0], word_ids):

    if word_idx is None:
        continue

    if word_idx != previous_word_idx:

        predicted_chunks.append(id2chunk[pred.item()])

    previous_word_idx = word_idx


for word, tag in zip(sentence.split(), predicted_chunks):

    print(word, "→", tag)

John → B-NP
works → B-VP
at → B-PP
Google → B-NP
in → B-PP
California → B-NP
